# Notebook to overplot MIRI photometry on existing PROSPECTOR fits

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import pandas as pd
import os

import scipy
import numpy as np
import astropy
import scipy.stats as stats

from astropy.table import Table
from prospector_utils.analysis import analyse_photspec_fits
from prospector_utils.plotting import *

quiescent = [7549, 8013, 8469, 9395, 10128, 10339, 10400, 10565, 10592, 11142, 11494, 16419, 18668, 21477]
no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
small_aperture = [7136, 1904, 7922, 8280, 8469, 10314, 11337, 11420, 11451, 12332, 17517, 17669, 18252, 18332, 21452]


What galaxies to exclude?

In [ ]:
table_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

table = Table.read(table_path, format='fits')
galaxy_ids = np.asarray([int(gid) for gid in table['ID']])

all_ids = set(galaxy_ids)

no_spec = set([9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990])
intermediate_ap = set([7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452])
bl_agn = set([12020, 18977])
high_z = set([1909, 11247, 7696, 21026])

exclude = no_spec | bl_agn | intermediate_ap | high_z

diff = all_ids.difference(exclude)
ids_to_exclude = all_ids.intersection(exclude)

print(f"Total number of MIRI IDs: {len(all_ids)}")

# Exclude IDs with no spectroscopic redshift confirmation
spec_discard = all_ids.intersection(no_spec)
print(f' - MIRI IDs with no spectroscopic redshift confirmation: {len(spec_discard)}')

all_ids.difference_update(no_spec)
print(f'MIRI IDs after excluding those with no spectroscopic redshift confirmation: {len(all_ids)}')

# Exclude IDs with insufficient aperture scaling
ap_discard = all_ids.intersection(intermediate_ap)
print(f' - MIRI IDs with intermediate aperture: {len(ap_discard)}')

all_ids.difference_update(intermediate_ap)
print(f'MIRI IDs after excluding those with intermediate aperture: {len(all_ids)}')

# Exclude IDs with broad-line AGN
agn_discard = all_ids.intersection(bl_agn)
print(f' - MIRI IDs with broad-line AGN: {len(agn_discard)}')

all_ids.difference_update(bl_agn)
print(f'MIRI IDs after excluding those with broad-line AGN: {len(all_ids)}')

# Exclude IDs of high-redshift filler targets
highz_discard = all_ids.intersection(high_z)
print(f' - MIRI IDs of high-redshift filler targets: {len(highz_discard)}')

all_ids.difference_update(high_z)
print(f'MIRI IDs after excluding those of high-redshift filler targets: {len(all_ids)}')

print(f"\nI should exclude {len(ids_to_exclude)} galaxies from my analysis with Prospector.")
print(f"This leaves {len(diff)} galaxies in total.")

# Reconstruct PROSPECTOR fits and overlay MIRI + model photometry

In [ ]:
table_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

table = Table.read(table_path, format='fits')
galaxy_ids = np.asarray([str(gid) for gid in table['ID']])

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/fits/'
photspec = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.3/poly10_cat/'
stats = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/pickle_files/'
phot_table = '/Users/benjamincollins/University/Master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

# The model is not stored! This needs to be fixed
# Good thing I have all the model parameters

analyse_photspec_fits(phot_table=phot_table, 
                    data_dir=photspec, 
                    plot_dir=plot_dir, 
                    stats_dir=stats)


Get scaling for the spectra and photometry

In [ ]:
cat_path = '/Users/benjamincollins/Data/Bluejay/combined_catalog_v2.0.4.fits'

cat_table = Table.read(cat_path)

objid = 17517

mask = cat_table['ID'] == objid

#print(cat_table['slit_flux_fraction_F444W'])
result = cat_table['slit_flux_fraction_F444W'][mask]

print(result.value[0])

#print(cat_table['slit_flux_fraction_F444W'])

Plot Prospector fit

In [ ]:
filename = f"/Users/benjamincollins/University/Master/Red_Cardinal/prospector/appphot_only_wMIRI/pickle_files/10128.pkl"

plot_dir = "/Users/benjamincollins/University/Master/Red_Cardinal/Paper/Figures"

with open(filename, 'rb') as f:
    fit_data = pkl.load(f)

gid = fit_data['id']
zred = fit_data['zred']

model = fit_data['model']
spec_best = model['spec_best']
spec_16th = model['spec_16th']
spec_median = model['spec_median']
spec_84th = model['spec_84th']
wave_spec = model['wave_spec']
sample_specs = model['sample_specs']
phot = model['phot']
phot_miri = model['phot_miri']
phot_miri_err = model['phot_miri_err']
phot_wave = model['phot_wave']
phot_wave_miri = model['phot_wave_miri']

obs = fit_data['obs']
maggies_to_muJy = fit_data['maggies_to_muJy']

# Convert to µJy
lower_scaled = spec_16th * maggies_to_muJy    
median_scaled = spec_median * maggies_to_muJy
upper_scaled = spec_84th * maggies_to_muJy
spec_scaled = spec_best * maggies_to_muJy

phot_wave_microns = phot_wave * 1e-4  # convert to µm
phot_wave_miri_microns = phot_wave_miri * 1e-4  # convert to µm

phot_scaled = phot * maggies_to_muJy
phot_miri_scaled = phot_miri * maggies_to_muJy
phot_miri_err_scaled = phot_miri_err * maggies_to_muJy

wave_spec_rs = wave_spec * 1e-4 * (1+zred)

# Initialise the plot
fig, ax = plt.subplots(figsize=(6.8, 4.3))

# Plot shaded region for 1σ uncertainty
ax.fill_between(wave_spec_rs, lower_scaled, upper_scaled, color='crimson', alpha=0.2, label='1σ uncertainty')

for spec in sample_specs:
    ax.plot(wave_spec_rs, spec*maggies_to_muJy, color='crimson', alpha=0.15, lw=0.8)

#ax.plot(wave_spec_rs, lower_scaled, color='blue', lw=0.8, label='16th percentile')
#ax.plot(wave_spec_rs, upper_scaled, color='blue', lw=0.8, label='84th percentile')
#########       PLOT THE BEST FIT      #########

ax.plot(wave_spec_rs, spec_scaled, '-', color='crimson', alpha=0.8, lw=1.5, label='Best-fit model')

#########    PLOT MODEL PHOTOMETRY     #########

ax.plot(phot_wave_microns, phot_scaled, 'd', markersize=6, color='black', label='Model photometry')
ax.errorbar(phot_wave_miri_microns, phot_miri_scaled, yerr=phot_miri_err_scaled, fmt='d', markersize=6, color='blue')
#ax.plot(phot_wave_miri_microns, phot_miri_scaled, 'd', markersize=6, color='black')

#########  PLOT MEASURED PHOTOMETRY    #########

# Define the style per instrument
instrument_styles = {
    'acs':     {'color': 'royalblue',   'marker': 'o', 'edgecolor': 'black', 'label': 'HST/ACS', 'ms': 10},
    'wfc3':    {'color': 'limegreen',  'marker': 'o', 'edgecolor': 'black', 'label': 'HST/WFC3', 'ms': 10},
    'nircam':  {'color': 'orange', 'marker': 'p', 'edgecolor': 'black',    'alpha': 0.7, 'label': 'JWST/NIRCam', 'ms': 10},
    'miri':    {'color': 'firebrick',    'marker': 'p', 'edgecolor': 'black',    'alpha': 0.7, 'label': 'JWST/MIRI', 'ms': 10}
}

# Get current labels to prevent duplicates
_, labels = ax.get_legend_handles_labels()

for i, filt in enumerate(obs['filters']):
        
        wave = obs['phot_wave'][i] * 1e-4  # convert to µm
        flux = obs['maggies'][i] * maggies_to_muJy  # µJy
        err  = obs['maggies_unc'][i] * maggies_to_muJy  # µJy
        
        name = filt.name.lower()     

        # Improved Upper Limit Logic for MIRI
        uplims = False
        
        if (flux / err < 3.0):
            uplims = True
            flux = 3 * err # Plot at 3-sigma
            err = flux * 0.4 # Small arrow size for visualisation
        
        if 'acs_wfc' in name:
            style = instrument_styles['acs']
        elif 'wfc3_ir' in name:
            style = instrument_styles['wfc3']
        elif 'miri' in name or any(m in name for m in ['f770w', 'f1000w', 'f1800w', 'f2100w']):
            style = instrument_styles['miri']            
        elif 'nircam' in name or ('jwst' in name and 'f' in name and 'w' in name):
            style = instrument_styles['nircam']
        else:
            continue  # skip unknown filters        

        ax.errorbar(
            wave, flux, yerr=err,
            fmt=style['marker'],
            color=style['color'],
            markeredgecolor=style.get('edgecolor', 'none'),
            alpha=style.get('alpha', 1.0),
            markersize=10,
            uplims=uplims, # This creates the actual downward arrow
            label=style['label'] if style['label'] not in labels else None
        )
        
        # Update labels list to prevent duplicates in current loop
        if style['label'] not in labels:
            labels.append(style['label'])

# Compute bounds
wave_mask = (wave_spec_rs >= 0.4) & (wave_spec_rs <= 35)

# Apply mask to spectrum(s)
spec_within = spec_scaled[wave_mask]  # works for 1D or 2D (e.g. percentiles)
spec_within = [ele for ele in spec_within if ele > 0]

# Compute y-axis limits
ymin = np.nanmin(spec_within)
ymax = np.nanmax(spec_within)

# Add margin proportionally, protecting against log-scale issues
ymin_plot = ymin * 0.2  # reduce, but stay > 0
ymax_plot = ymax * 5   # increase

# Set limits
ax.set_ylim(ymin_plot, ymax_plot)

# Plot formatting
ax.set_xlabel('Observed Wavelength [µm]', fontsize=14)
ax.set_ylabel('Flux [µJy]', fontsize=14)
ax.set_xlim(0.4, 35)#200)    # Change x range    
ax.set_xscale('log')
ax.set_yscale('log')

ax.legend(loc="lower right", fontsize=12)

#ax.set_title(f"Galaxy {objid} at z={np.round(zred,2)}", fontsize=14)

ax.tick_params(axis='both', which='major', labelsize=14)

zred_rounded = np.round(zred,2)
#plt.title(f"Galaxy {objid} at z={zred_rounded}")
plt.tight_layout()

os.makedirs(plot_dir, exist_ok=True)
fname = os.path.join(plot_dir, f'{gid}_wspec_paper.png')
plt.savefig(fname)
print(f"Plot saved to {fname}")
plt.show()
plt.close()

In [ ]:
for gid in galaxy_ids:
    filename = f"/Users/benjamincollins/University/Master/Red_Cardinal/prospector/pickle_files/{gid}.pkl"

    plot_reconstructed_fit(filename, plot_dir)

# Plot NSigma and R distributions

Prepare the data

In [ ]:
pickle_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/pickle_files/'
plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/fit_quality/'

os.makedirs(plot_dir, exist_ok=True)

pickle_files = glob.glob(f'{pickle_dir}/*.pkl')

all_data = []
reduced_chi2_list = []

n_gals = 0

# 1. DATA EXTRACTION
for filename in pickle_files:
    with open(filename, 'rb') as f:
        data = pkl.load(f)
        
    gid = data['id']
    
    if gid in ids_to_exclude:
        continue
    
    fit_quality = data.get('fit_quality', {})
    
    # Store global per-galaxy stats
    if 'chi2_red' in fit_quality:
        reduced_chi2_list.append({
            'galaxy_id': gid,
            'reduced_chi2': fit_quality['chi2_red'],
            'n_filters': len(fit_quality) - 1 # Assuming only 'chi2_red' is non-filter
        })

    # Store per-band stats
    for i, (key, val) in enumerate(fit_quality.items()):
        if isinstance(val, dict): # This identifies the filter entries
            all_data.append({
                'galaxy_id': gid,
                'filter_name': key,
                'N_sigma': val.get('n_sigma')*(-1),
                'flux': val.get('obs_flux'),
                'flux_err': val.get('obs_err'),
                'flux_mod': val.get('mod_flux'),
                'flux_mod_err': val.get('mod_err'),
                'frac_diff': val.get('frac_diff')*(-1),
                'snr': val.get('snr')
            })
            
            if val.get('n_sigma') > 7.0:
                print(gid, key)
                print(np.log10(val.get('obs_flux')))
                print(np.log10(val.get('mod_flux')))
                print("Ratio:", np.log10(val.get('obs_flux')/val.get('mod_flux')))
                print("Nsigma:", val.get('n_sigma'))
                print("\n")
    n_gals += 1
df = pd.DataFrame(all_data)
chi2_df = pd.DataFrame(reduced_chi2_list)

print(n_gals)

bands = ['F770W', 'F1000W', 'F1800W', 'F2100W']
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']  # Distinct colors per band

Plot N Sigma distributions

In [ ]:
# PLOT 1: N_SIGMA HISTOGRAMS
#fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=False, sharey=True)
fig, axes = plt.subplots(2, 2, figsize=(8, 6.5), sharex=False, sharey=True)
axes = axes.flatten()  # easier to index

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    nsigmas = subset['N_sigma']
    if len(nsigmas) == 0: continue
    
    ax.set_title(f'{band}')
    #ax.set_xlim(x_min, x_max)
    ax.set_xlabel(r'$\mathrm{N_\sigma}$', fontsize=12)
    ax.set_ylabel('Number of galaxies', fontsize=12)
    
    #if i in [0,1]: ax.set_ylim(0, 24)
    #elif i in [2,3]: ax.set_ylim(0,12)

    # Add compact statistics
    mean_ratio = np.mean(nsigmas)
    median_ratio = np.median(nsigmas)
    std_ratio = np.std(nsigmas)
    mad_ratio = median_abs_deviation(nsigmas)
    N = len(subset['galaxy_id'].unique())
    num = f'N = {N}'
    
    x_min = -8.5
    x_max = 8.5
    bins = np.linspace(x_min, x_max, 25)
    
    counts, bin_edges, _ = ax.hist(nsigmas, bins=bins, color=colors[i], alpha=0.7, edgecolor='black')

    x = np.linspace(x_min, x_max, 500)
    gaussian_norm = stats.norm.pdf(x, loc=0, scale=1)
    gaussian_obs = stats.norm.pdf(x, loc=median_ratio, scale=mad_ratio)

    # Scale Gaussians to match histogram counts
    gaussian_norm_scaled = gaussian_norm * len(nsigmas) * (bin_edges[1] - bin_edges[0])
    gaussian_obs_scaled = gaussian_obs * len(nsigmas) * (bin_edges[1] - bin_edges[0])
    
    ax.plot(x, gaussian_norm_scaled, 'gray', lw=2, alpha=1, label=r'$\mathcal{N}(0,1)$')
    ax.plot(x, gaussian_obs_scaled, colors[i], lw=2, alpha=1, label=r'$\mathcal{N}_{\mathrm{obs}}$')
    
    ax.vlines(0.0, ymin=0, ymax=25, color='black', alpha=0.6, linestyle='--', linewidth=2)
    ax.vlines(median_ratio, ymin=0, ymax=25, color=colors[i], alpha=0.6, linestyle='--', linewidth=2)
    
    median_ratio = np.median(nsigmas)
    
    stats_text = f'Med = {median_ratio:.2f}\nMAD = {mad_ratio:.2f}\n{num}'
    ax.legend(loc="upper right")
    #ax.text(0.78, 0.84, stats_text, transform=ax.transAxes, fontsize=10,
    #        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax.text(0.03, 0.76, stats_text, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    # Annotate in the top-right corner (adjust x,y if needed)
    #ax.text(0.95, 0.95, f'N = {n_galaxies}', 
    #        transform=ax.transAxes, ha='right', va='top',
    #        fontsize=10, bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
    
#plt.suptitle(r'$N_\sigma$ distribution for each MIRI filter', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
# Save single combined figure
filename = os.path.join(plot_dir, 'Nsigma_gauss.png')
plt.savefig(filename, dpi=300)
plt.show()

Plot the log-ratios

In [ ]:
# PLOT 2: LOG RATIOS
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)
axes = axes.flatten()  # easier to index

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    
    f_obs = subset['flux']
    snr = subset['snr']
    f_mod = subset['flux_mod']
    
    # SNR filter: Keep only detections > 3-sigma
    snr_mask = snr >= 3.0
    
    log_ratios = np.log10(f_mod[snr_mask]/f_obs[snr_mask])
    
    ax.set_title(f'{band}')
    #ax.set_xlim(x_min, x_max)
    ax.set_xlabel('Flux ratio (dex)')
    ax.set_ylabel('Number of galaxies')
    
    #if i in [0,1]: ax.set_ylim(0, 24)
    #elif i in [2,3]: ax.set_ylim(0,12)
    
    # Add compact statistics
    mean_logr = np.mean(log_ratios)
    std_logr = np.std(log_ratios)
    median_logr = np.median(log_ratios)
    mad_logr = median_abs_deviation(log_ratios)
    N = len(log_ratios)
    num = f'N = {N}'
    
    x_min = -1.2
    x_max = 1.2
    bins = np.linspace(x_min, x_max, 25)
    
    counts, _, _ = ax.hist(log_ratios, bins=bins, color=colors[i], alpha=0.7, edgecolor='black')

    ymax = np.max(counts) * 1.1 # for all plots
    ymax = max(ymax, 10)
    ax.set_ylim(0,ymax)
    ax.vlines(median_logr, ymin=0, ymax=ymax, color='darkred', alpha=0.8, linestyle='-', linewidth=2, label=f'Median: {median_logr:.2f}')
#            if i == 2: 
#               stats_text += ' (*)'
#              print(log_ratios[log_ratios > 1])
    ax.text(0.025, 0.92, num, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.plot([],[], label=f'MAD: {mad_logr:.2f}', alpha=0)  # dummy plot for legend
    ax.legend()
    
#plt.suptitle(r'$N_\sigma$ distribution for each MIRI filter', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
# Save single combined figure
filename = os.path.join(plot_dir, 'log_ratios.png')
plt.savefig(filename, dpi=300)
plt.show()

Plot the reduced Chi squared

In [ ]:
# Compute reduced chi^2 per galaxy

chi2_red = chi2_df['reduced_chi2']

# --- 1. Clipping & Statistics ---
chi2_red = chi2_df['reduced_chi2']
q95 = chi2_red.quantile(0.95)
filtered = chi2_df[chi2_red <= q95]

mean_val = chi2_red.mean()
median_val = chi2_red.median()
mad_val = median_abs_deviation(chi2_red.dropna())

fig, axes = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={"width_ratios":[1.25,0.75]})

# --- Left: Nsigma vs sSFR
counts, bins, patches = axes[0].hist(filtered['reduced_chi2'], bins=25, alpha=0.7, edgecolor='black', range=(0, q95))

axes[0].set_xlabel(r'Reduced $\chi^2$')
axes[0].set_ylabel('Number of galaxies')

# Count how many chi2 values are in the histogram
chi2_values = len(chi2_df)
num = f'\nN = {len(filtered)}/{chi2_values}\n(95th perctile)'
# Annotate in the top-right corner (adjust x,y if needed)

# plot vertical lines for mean and median
# Plot vertical lines for mean and median
ymax = counts.max() * 1.1
axes[0].vlines(mean_val, ymin=0, ymax=ymax, color='red', alpha=0.8, linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
axes[0].vlines(median_val, ymin=0, ymax=ymax, color='darkred', alpha=0.8, linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')
axes[0].plot([],[], label=num, alpha=0)  # dummy plot for legend
axes[0].set_ylim(0, ymax)
axes[0].legend()    


# Scatter plot with filtered data
#plt.scatter(filtered['n_filters'], filtered['reduced_chi2'], alpha=0.7)    
axes[1].scatter(filtered['n_filters'], filtered['reduced_chi2'], alpha=0.7)    
axes[1].set_xlabel('Number of photometric data points')
axes[1].set_ylabel(r'Reduced $\chi^2$')
#plt.title(r'Reduced $\chi^2$ vs. number of MIRI bands')
axes[1].axhline(1, color='orange', linestyle='--', label='Unity')
axes[1].legend()    

plt.tight_layout()
filename = os.path.join(plot_dir, 'reduced_chi2_big.png')
plt.savefig(filename, dpi=300)
plt.show()

threshold = 30  # user-specified value
high_chi2_ids = chi2_df.loc[chi2_df['reduced_chi2'] > threshold, 'galaxy_id'].tolist()
print(f"✅ Saved reduced chi^2 plots to {filename}")
print(f"{len(high_chi2_ids)} galaxies have reduced χ² > {threshold}")
print("These galaxies are:", high_chi2_ids)
print("Their χ² values are:", chi2_df.loc[chi2_df['reduced_chi2'] > threshold, 'reduced_chi2'].tolist())

Now one plot showing the flux ratios of model and error

In [ ]:
# PLOT: LOG FLUX MODEL vs LOG FLUX OBS (1 row, 4 panels)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=False, sharey=False)

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    f_obs = subset['flux']
    err_obs = subset['flux_err']
    f_mod = subset['flux_mod']
    err_mod = subset['flux_mod_err']
    frac_diff = subset['frac_diff']
    snr = subset['snr']

    # SNR filter + require both fluxes to be positive
    valid_mask = (snr >= 3.0) & (f_obs > 0) & (f_mod > 0) & (err_obs > 0) & (err_mod > 0)

    f_obs_v = f_obs[valid_mask].values
    f_mod_v = f_mod[valid_mask].values
    
    # CORRECT - propagated uncertainty in log space
    err_obs_v = err_obs[valid_mask].values
    err_mod_v = err_mod[valid_mask].values
    
    
    
    N = valid_mask.sum()

    # 1. Split your data into "Normal" and "Upper Limit" groups
    is_uplim = (err_mod_v > 10 * err_obs_v) & (f_mod_v - err_mod_v < 0)
    norm = ~is_uplim

    # 2. Plot Normal points as usual
    ax.errorbar(f_obs_v[norm], f_mod_v[norm], 
                xerr=err_obs_v[norm], yerr=err_mod_v[norm],
                fmt='o', alpha=0.7, capsize=2, markersize=6, color=colors[i])

    # 3. Plot Upper Limits starting from the TOP of the error bar
    if is_uplim.any():
        # The 'y' position is now the top of the error bar
        y_top = f_mod_v[is_uplim] + err_mod_v[is_uplim]
        
        # Define how long the arrow should be (e.g., 20% of the top value for a tidy look)
        arrow_length = y_top * 0.3 
        
        ax.errorbar(f_obs_v[is_uplim], y_top, 
                    xerr=err_obs_v[is_uplim], 
                    yerr=arrow_length,
                    uplims=True, 
                    fmt='none', # Don't plot a new marker at the top
                    color=colors[i], alpha=0.7, capsize=0)
        
        # 4. Optional: Plot the original central point marker so you can still see the best fit
        #ax.scatter(f_obs_v[is_uplim], f_mod_v[is_uplim], 
        #        marker='o', s=36, color=colors[i], alpha=0.7, edgecolors='none')
    
    ax.loglog()  # switches to log scale with physical tick values

    # 1:1 line in flux space
    lims_low  = min(f_obs_v.min(), f_mod_v.min()) * 0.6
    lims_high = max(f_obs_v.max(), f_mod_v.max()) * 1.4
    ax.plot([lims_low, lims_high], [lims_low, lims_high], 'k--', linewidth=1, label='1:1')
    ax.set_xlim(lims_low, lims_high)
    ax.set_ylim(lims_low, lims_high)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=16)

    ax.set_title(band, fontsize=20)
    ax.set_xlabel(r'$f_\mathrm{obs}$ [µJy]', fontsize=20)
    if i == 0:
        ax.set_ylabel(r'$f_\mathrm{model}$ [µJy]', fontsize=20)

    ax.legend(fontsize=14, loc='upper left')
    #ax.grid(True, alpha=0.3)

plt.tight_layout()
filename = os.path.join(plot_dir, 'log_flux_model_vs_obs_uplims.png')
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()

# Read the new Blue Jay table

In [ ]:
cat_path = '/Users/benjamincollins/Data/Bluejay/combined_catalog_v2.0.4.fits'
cat_table = Table.read(cat_path)

#print(cat_table.colnames)

columns_to_keep = [
    'ID', 
    'z_spec', 
    'Balmer_dec', 
    'Balmer_dec_err', 
    'dust2_spec', 
    'dust2_spec_err', 
    'dust1_fraction_spec',
    'Av_gas_spec' # This might already be a Prospector-inferred value to check against!
]

df = cat_table[columns_to_keep].to_pandas()

pickle_file = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/pickle_files/7102.pkl'

with open(pickle_file, 'rb') as f:
    data = pkl.load(f)

#print(df)
#print(data['map_theta']['dust2'])
#print(data['map_theta']['dust1_fraction'])
#print(data['map_theta']['zred'])

balmer_dec = df['Balmer_dec']
id = df['ID']
av_gas = df['Av_gas_spec']

# Your input data
bd = np.array(balmer_dec) 
av_gas = np.array(av_gas)

# 1. Create a mask for valid data (remove the -99 flags and any zeros)
valid_mask = (bd > 0) & (av_gas > 0)

# 2. Setup an array for Av (default to NaN or 0)
av_balmer = np.full_like(bd, np.nan)


Rv = 4.05   # Empirical Calzetti value for starforming galaxies
lam_alpha = 0.6563
k_alpha = 2.659 * (-1.857 + 1.040/lam_alpha) + Rv
print(k_alpha)

lam_beta = 0.4861
k_beta = 2.659 * (-2.156 + 1.509/lam_beta - 0.198/lam_beta**2 + 0.011/lam_beta**3) + Rv
print(k_beta)

# 3. Calculate only for valid entries
# Use (BD / 2.86) so that BD > 2.86 results in positive Av
colour_excess = (2.5/(k_beta - k_alpha)) * np.log10(bd[valid_mask]/2.86)

av_balmer[valid_mask] = 4.05 * colour_excess

# 4. Handle the "Bluer than theoretical" case
# If BD is between 0 and 2.86, Av will be negative. 
# Physically, we clip these to 0.0.
av_balmer[valid_mask & (av_balmer < 0)] = np.nan

#print(av_balmer)

av_dict = {}
for gid, avb, avg in zip(id, av_balmer, av_gas):
    if not np.isnan(avb) and not np.isnan(avg):
        av_dict[gid] = avb, avg

# 1. Calculate the Prospector reconstructed Av for the gas
df['Av_prospector_reconstructed'] = 1.086 * df['dust2_spec'] * (1 + df['dust1_fraction_spec'])

# 2. Add your calculated Balmer Av to the dataframe for easy comparison
# We can map it using the ID to ensure rows match
df['Av_balmer_calc'] = df['ID'].map(av_dict)

# 3. Quick comparison check
comparison = df[['ID', 'Av_gas_spec', 'Av_prospector_reconstructed', 'Av_balmer_calc']].head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

pickle_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/pickle_files/'
plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/fit_quality/'

pickle_files = glob.glob(f'{pickle_dir}/*.pkl')

# 1. Prepare the catalog for fast lookup
# We set 'ID' as the index so we can call df_indexed.loc[id]
df_indexed = df.set_index('ID')

# Data storage for the comparison
results = {
    'id': [],
    'av_prosp_reconstructed': [],
    'av_gas_spec_catalog': [],
    'diff': []
}

for f_path in pickle_files:
    with open(f_path, 'rb') as f:
        data = pkl.load(f)
    
    gid = data.get('id')
    
    # Check if this ID exists in both your Balmer dict AND the catalog
    if gid not in av_dict or gid not in df_indexed.index:
        continue

    map_params = data.get('map_theta', {})
    dust2 = map_params.get('dust2', np.nan)
    dust1_fraction = map_params.get('dust1_fraction', np.nan)
    
    # 1. Your reconstructed value from the pickle
    dust_prosp_recon = 1.086 * dust2 * (1 + dust1_fraction)
    
    # 2. The value directly from Sirio's catalog entry
    dust_catalog_val = df_indexed.loc[gid, 'Av_gas_spec']
    
    # Store results
    results['id'].append(gid)
    results['av_prosp_reconstructed'].append(dust_prosp_recon)
    results['av_gas_spec_catalog'].append(dust_catalog_val)
    results['diff'].append(dust_prosp_recon - dust_catalog_val)

# Convert to arrays for plotting
x_recon = np.array(results['av_prosp_reconstructed'])
y_cat = np.array(results['av_gas_spec_catalog'])

# 2. Plotting the Consistency Check
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(x_recon, y_cat, alpha=0.6, edgecolors='k', label='Galaxies')

# 1:1 Line
lims = [0, max(max(x_recon), max(y_cat))]
ax.plot(lims, lims, 'r--', alpha=0.8, label='1:1 Line')

ax.set_xlabel(r'Reconstructed $A_V$ ($1.086 \cdot \tau_2 \cdot (1+f_{d1})$)', fontsize=12)
ax.set_ylabel(r'Catalogue `Av_gas_spec`', fontsize=12)
ax.set_title('Prospector Consistency Check', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Now let's plot the comparison!

In [ ]:
from scipy.stats import binned_statistic

pickle_dir = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/photspec/pickle_files'
out_dir='/Users/benjamincollins/University/Master/Red_Cardinal/prospector/photspec/fit_quality/'

os.makedirs(out_dir, exist_ok=True)
pickle_files = glob.glob(f'{pickle_dir}/*.pkl')

d = {'nsig': [], 'ssfr': [], 'dust': []}
q = {'nsig': [], 'ssfr': [], 'dust': []}    # For quiescent galaxies

for f_path in pickle_files:
    with open(f_path, 'rb') as f:
        data = pkl.load(f)
    
    id = data.get('id', {})
    
    if id not in av_dict.keys():
        continue

    fq = data.get('fit_quality', {})
    props = data.get('galaxy_properties', {})
    
    # Pre-calculate common parameters to save CPU cycles
    log_m = props.get('logmass', np.nan)
    sfr = props.get('sfr_100myr', np.nan)
    ssfr = np.log10(sfr / 10**log_m) if log_m and sfr else np.nan
    
    map_params = data.get('map_theta', {})
    dust2 = props.get('dust2', np.nan)
    dust1_fraction = map_params.get('dust1_fraction', np.nan)
    
    dust_prosp = 1.086 * dust2 * (1 + dust1_fraction)
    
    dust_balmer = av_dict[id][0]
    dust_gas = av_dict[id][1]
    
    dust_val = dust_balmer - dust_prosp
    #dust_val = dust_balmer - dust_gas
    
    if 'F2100W' in fq:
        id = fq['F2100W']['galaxy_id']
        # Exclude quiescent galaxies of Bugiani et al. (2025)
        if id in no_spec:
            continue
        
        # Exclude all galaxies with small apertures
        if id in small_aperture:
            continue
        
        # Inside your loop
        if dust_balmer > 7 or dust_balmer < -2: # Extreme outlier cut
            continue
        
        if id in quiescent or id in below_ms:
            q['nsig'].append(fq["F2100W"]['n_sigma']*(-1))
            q['ssfr'].append(ssfr)
            q['dust'].append(dust_val)
            
        else: 
            d['nsig'].append(fq["F2100W"]['n_sigma']*(-1))
            d['ssfr'].append(ssfr)
            d['dust'].append(dust_val)

# 2. Setup Plotting
fig, ax = plt.subplots(figsize=(6, 4))

# Convert to numpy arrays for the specific band
x = np.array(d['dust'])
y = np.array(d['nsig'])
c = np.array(d['ssfr'])

# Quiescent galaxies
xq = np.array(q['dust'])
yq = np.array(q['nsig'])
cq = np.array(q['ssfr'])

xall = np.concatenate((x, xq))
yall = np.concatenate((y, yq))
# Bin the data into 10 bins (adjust as needed)
n_bins = 5
bin_edges = np.linspace(xall.min(), xall.max(), n_bins + 1)

# Compute the running mean
#mean, bin_edges, _ = binned_statistic(
#    xall, yall, statistic='mean', bins=bin_edges
#)

# Compute the bin centers for plotting
#bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Plot the running median
#ax.plot(bin_centers, mean, color='black', lw=2, alpha=0.8, label='Running Mean')

#print(mean[0])
#print(mean[-1])

ax.scatter(xq, yq, alpha=0.7, color='grey', s=55, label='quiescent')


sc = ax.scatter(x, y, c=c, cmap='plasma', 
                alpha=0.9, edgecolor='black', s=55)


# Labels and Style
ax.axhline(0, ls="--", c="grey", alpha=0.5)
#ax.set_title(r"F2100W: $N_\sigma$ vs $\mathrm{A_V}$", fontsize=14)
ax.set_ylabel(r"$N_\sigma$", fontsize=15)
ax.set_ylim(-6, 6)
ax.set_xlabel(r"$A_{V,Balmer} - A_{V,Prospector}$", fontsize=15)
#ax.set_xlim(-0.1, 3.0)

ax.tick_params(axis='both', which='major', labelsize=14)
ax.legend(fontsize=12)

# Colorbar
cb = fig.colorbar(sc, ax=ax)
cb.set_label(r"$\log(\mathrm{sSFR}_{100})$", fontsize=15)
cb.ax.tick_params(labelsize=14)

# Count label
#ax.text(0.03, 0.92, f'N = {np.sum(mask)}', transform=ax.transAxes, 
#        fontsize=13, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.title("F2100W", fontsize=17)
plt.tight_layout()
filename = os.path.join(out_dir, 'nsigma_vs_dust_balmer_prosp.png')
plt.savefig(filename, dpi=300, bbox_inches='tight')
print(f"✅ Plot saved as {filename}")
plt.show()


# Median and MAD R vs data

In [ ]:
sourcephotonly = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/sourcephotonly/flux_scaled/pickle_files'
photspec = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/photspec/flux_scaled/pickle_files'
appphot_only_wMIRI = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/pickle_files'

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation

# Paths to your directories
dirs = {
    'HST/NIRCam': sourcephotonly,
    'HST/NIRCam + NIRSpec': photspec,
    'HST/NIRCam + MIRI': appphot_only_wMIRI
}

results = {}

for label, path in dirs.items():
    band_data = {'F770W': [], 'F1000W': [], 'F1800W': [], 'F2100W': []}    
    # Iterate through every pickle file in the directory
    for file in os.listdir(path):
        if file.endswith('.pkl'):
            with open(os.path.join(path, file), 'rb') as f:
                data = pickle.load(f)
            
            fit_quality = data.get('fit_quality', {})

            # Store per-band stats
            for band, val in fit_quality.items():
                
                if isinstance(val, dict): # This identifies the filter entries
                    obs_flux = val.get('obs_flux')
                    mod_flux = val.get('mod_flux')
                    snr = val.get('snr')

                    if snr > 3.0:
                        # Compute log ratio: log10(F_obs / F_model)
                        ratio = np.log10(mod_flux/obs_flux)
                        band_data[band].append(ratio)   
                         
    # Calculate stats per band for this directory
    results[label] = {
        band: {
            'median': np.nanmedian(ratios),
            'mad': median_abs_deviation(ratios, nan_policy='omit')
        } for band, ratios in band_data.items()
    }
    
    print(results[label])


Now let's plot

In [ ]:
all_bands = results['HST/NIRCam'].keys()


fig, ax = plt.subplots(figsize=(5.3, 3.5))
colors = ['#4C72B0', '#55A868', '#C44E52'] # Blue, Red, Green
offsets = [-0.2, 0, 0.2] # Shift markers slightly so they don't overlap

for (label, stats), color, offset in zip(results.items(), colors, offsets):
    x_pos = np.arange(len(all_bands)) + offset
    
    # Get medians/mads only for bands that exist in this specific result
    y_vals = [stats[b]['median'] if b in stats else np.nan for b in all_bands]
    y_errs = [stats[b]['mad'] if b in stats else np.nan for b in all_bands]
    
    print(y_vals)
    print(y_errs)
    
    ax.errorbar(x_pos, y_vals, yerr=y_errs, fmt='o', label=label, 
                color=color, capsize=3.5, alpha=0.8, ms=6)

ax.set_xticks(range(len(all_bands)))
ax.set_xticklabels(all_bands, rotation=45)
ax.tick_params(labelsize=12)
ax.set_ylim(-1.1, 1.1)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_ylabel(r'$\log_{10}(f_{model}/f_{obs})$', fontsize=15)
ax.legend(fontsize=11, loc='lower left')
fig.tight_layout()
plt.savefig('/Users/benjamincollins/University/Master/Red_Cardinal/prospector/fit_improvement_scaled.png', dpi=300)
plt.show()

# Comparison between the galaxy parameters with and without NIRSpec!

In [ ]:
with_spec = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/pickle_files/'
no_spec = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/flux_scaled/pickle_files/'

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/parameter_comparisons/'
os.makedirs(plot_dir, exist_ok=True)

def extract_summary_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Extract the scalar properties we want to compare
            # We flatten the 'galaxy_properties' nested dict here
            entry = {
                'id': data['id'],
                'zred': data['zred'],
                **data['galaxy_properties'] # Unpacks logmass, dust2, sfr_100myr, etc.
            }
            
            if data['id'] in ids_to_exclude:
                #print("Skipping galaxy...")
                continue
            
            summary_list.append(entry)
            
    return pd.DataFrame(summary_list)

# 1. Define your paths (update these to your actual folder/naming convention)
path_no_spec = glob.glob(os.path.join(no_spec,'*.pkl'))
path_with_spec = glob.glob(os.path.join(with_spec,'*.pkl'))

# 2. Extract into DataFrames
df_no_spec = extract_summary_data(path_no_spec)
df_with_spec = extract_summary_data(path_with_spec)

# 3. Merge the two datasets on 'id'
# We add suffixes so we can tell which logmass is which
df_comparison = pd.merge(
    df_no_spec, 
    df_with_spec, 
    on='id', 
    suffixes=('_no', '_with')
)

print(f"Successfully merged data for {len(df_comparison)} galaxies.")

print(df_comparison.head())

Plot the dust parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle as pkl
import glob
import numpy as np
import scipy.stats as stats

def calculate_statistics(x, y, valid_mask):
    """
    Calculate comparison statistics
    """
    if np.sum(valid_mask) < 3:
        return {}
    
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    
    # Linear correlation
    corr_coef, p_value = stats.pearsonr(x_valid, y_valid)
    
    # Calculate residuals and statistics
    residuals = y_valid - x_valid
    mean_residual = np.mean(residuals)
    median_residual = np.median(residuals)
    std_residual = np.std(residuals)
    rms_residual = np.sqrt(np.mean(residuals**2))
    
    # Fractional differences for positive values
    frac_diff = (y_valid - x_valid) / x_valid
    median_frac_diff = np.median(frac_diff)
    mean_frac_diff = np.mean(frac_diff)
    std_frac_diff = np.std(frac_diff)
    
    return {
        'correlation': corr_coef,
        'p_value': p_value,
        'median_residual': median_residual,
        'mean_residual': mean_residual,
        'std_residual': std_residual,
        'rms_residual': rms_residual,
        'median_frac_diff': median_frac_diff,
        'mean_frac_diff': mean_frac_diff,
        'std_frac_diff': std_frac_diff,
        'n_objects': len(x_valid)
    }

def extract_dust_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Helper function to find parameter in either dict
            def get_param(name):
                if name in data['galaxy_properties']:
                    return data['galaxy_properties'][name]
                elif name in data['map_theta']:
                    return data['map_theta'][name]
                return np.nan

            entry = {
                'id': data['id'],
                'dust1_fraction': get_param('dust1_fraction'),# * get_param('dust2'),
                'dust_index': get_param('dust_index'),
                'dust2': get_param('dust2')
            }
            
            if data['id'] in ids_to_exclude:
                continue
            
            summary_list.append(entry)
    return pd.DataFrame(summary_list)

# 1. Update these paths to your actual folders
path_no_spec = glob.glob(os.path.join(no_spec,'*.pkl'))
path_with_spec = glob.glob(os.path.join(with_spec,'*.pkl'))

# 2. Extract and Merge
df_no = extract_dust_data(path_no_spec)
df_with = extract_dust_data(path_with_spec)
df = pd.merge(df_no, df_with, on='id', suffixes=('_no', '_with'))

# 3. Create the 3x1 Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
dust_params = [
    ('dust1_fraction', 'dust1_fraction', '#4C72B0'),
    ('dust_index', 'dust_index', '#55A868'),
    ('dust2', 'dust2', '#C44E52')
]

for i, (col, label, colour) in enumerate(dust_params):
    ax = axes[i]
    
    x = df[f'{col}_with']
    y = df[f'{col}_no']
        
    corr_coef, p_value = stats.pearsonr(x, y)
    
    ax.scatter(x, y, alpha=0.6, edgecolors='black', s=45, color=colour)
    
    if i==1: 
        ax.set_xlim(-0.75, 0.75)
        ax.set_ylim(-0.75, 0.75)
    
    # Force square aspect ratio so 1-to-1 actually looks like a 45-degree line
    ax.set_aspect('equal', adjustable='box')
    
    plt.draw() 
    ticks = ax.get_xticks()
    ax.set_yticks(ticks)
    
    #if i in [0, 1]:
    if i < 2:
        x_min = ticks[0] + 0.25
        x_max = ticks[-1]
    
        ax.set_xticks(np.arange(x_min, x_max, 0.5))
        ax.set_yticks(np.arange(x_min, x_max, 0.5))
    
        #ax.set_xticks(np.arange(-1, 1, 0.5))
        #ax.set_yticks(np.arange(-1, 1, 0.5))
    
    else:
        x_min = ticks[1]
        x_max = 5#ticks[-1] + 1
        
        ax.set_xticks(np.arange(x_min, x_max, 1))
        ax.set_yticks(np.arange(x_min, x_max, 1))
        ax.set_xlim(-0.2)
        ax.set_ylim(-0.2)
            
    # Add a 1-to-1 reference line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
    
    
    ax.set_title(f'{label}', fontsize=18)
    ax.set_xlabel('HST/NIRCam + NIRSpec', fontsize=16)
    if i==0: ax.set_ylabel('HST/NIRCam', fontsize=16)
    ax.tick_params(labelsize=14)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    ax.text(0.05, 0.95, f'r = {corr_coef:.3f}',
                transform=ax.transAxes, verticalalignment='top', fontsize=14,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))


plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/parameter_comparisons/'

#plt.suptitle("Dust Absorption Parameters", fontsize=17)
filename = os.path.join(plot_dir, 'dust_params_v5.png')

plt.tight_layout()
plt.savefig(filename, dpi=300)
print(f"Figure saved to {filename}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle as pkl
import glob
import numpy as np

def extract_dust_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Helper function to find parameter in either dict
            def get_param(name):
                if name in data['galaxy_properties']:
                    return data['galaxy_properties'][name]
                elif name in data['map_theta']:
                    return data['map_theta'][name]
                return np.nan

            entry = {
                'id': data['id'],
                'duste_umin': get_param('duste_umin'),
                'duste_qpah': get_param('duste_qpah'),
                'duste_gamma': get_param('duste_gamma')
            }
            
            if data['id'] in exclude:
                continue
            
            summary_list.append(entry)
    return pd.DataFrame(summary_list)

# 1. Update these paths to your actual folders
path_no_miri = glob.glob(os.path.join(no_spec,'*.pkl'))
path_with_miri = glob.glob(os.path.join(with_spec,'*.pkl'))

# 2. Extract and Merge
df_no = extract_dust_data(path_no_miri)
df_with = extract_dust_data(path_with_miri)
df = pd.merge(df_no, df_with, on='id', suffixes=('_no', '_with'))

# 3. Create the 3x1 Plot
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
dust_params = [
    ('duste_umin', r'$U_{min}$ ', '#4C72B0'),
    ('duste_qpah', 'PAH-fraction', '#55A868'),
    ('duste_gamma', r'Dust $\gamma$', '#C44E52')
]

for i, (col, label, colour) in enumerate(dust_params):
    ax = axes[i]
    
    x = df[f'{col}_with']
    y = df[f'{col}_no']
    
    ax.scatter(x, y, alpha=0.6, edgecolors='black', s=60, color=colour)
    
    # Add a 1-to-1 reference line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
    
    ax.set_title(f'{label}', fontsize=15)
    ax.set_xlabel('With NIRSpec', fontsize=13)
    if i == 0: ax.set_ylabel('Only HST/NIRCam', fontsize=13)
    ax.tick_params(labelsize=14)
    #ax.grid(True, linestyle=':', alpha=0.6)

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/photspec/flux_scaled/parameter_comparisons/'

#plt.suptitle("Dust Emission Parameters", fontsize=17)
filename = os.path.join(plot_dir, 'duste_params.png')

plt.tight_layout()
plt.savefig(filename, dpi=300)
print(f"Figure saved to {filename}")
plt.show()